# Tokenizing 
Starter notebook — select kernel **Python (tjabane.thuto.embedding)** before running.

In [308]:
import re
import torch
import tiktoken
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

In [309]:
data_dir = Path("data")
file_path = data_dir / "The_Verdict.txt"

with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

print(f"Loaded: {file_path} ({len(raw_text):,} characters)")

Loaded: data\The_Verdict.txt (21,728 characters)


In [310]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))

4916


## Vectorizing

In [311]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1217


In [312]:
word_to_index = {token: index for index, token in enumerate(all_words)}
index_to_word = {index: token for index, token in enumerate(all_words)}

In [313]:
class SimpleTokenizerV1:
    def __init__(self, word_to_index, index_to_word):
        self.word_to_index = word_to_index
        self.index_to_word = index_to_word

    def encode(self, text):
        text = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        text = [item.strip() for item in text if item.strip()]
        return [self.word_to_index[token] for token in text]

    def decode(self, indices):
        return " ".join([self.index_to_word[index] for index in indices])

In [314]:
tokenizer = SimpleTokenizerV1(word_to_index, index_to_word)
encoded = tokenizer.encode("The height of his glory")
print(encoded)
decoded = tokenizer.decode(encoded)
print(decoded)

[113, 593, 788, 604, 551]
The height of his glory


In [315]:
try:
    tokenizer.encode("This word does not exist in the vocabulary")
except KeyError as e:
    print(f"Error: {e} is not in the vocabulary")

Error: 'does' is not in the vocabulary


### Handle Unknown Words and EndOfDocument 

In [316]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|pad|>", "<|unk|>"])
word_to_index = {token: index for index, token in enumerate(all_tokens)}
index_to_word = {index: token for index, token in enumerate(all_tokens)}

In [317]:
class SimpleTokenizerV2:
    def __init__(self, word_to_index, index_to_word):
        self.word_to_index = word_to_index
        self.index_to_word = index_to_word

    def encode(self, text):
        text = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        text = [item.strip() for item in text if item.strip()]
        return [self.word_to_index.get(token, self.word_to_index["<|unk|>"]) for token in text]

    def decode(self, indices):
        return " ".join([self.index_to_word[index] for index in indices])

In [318]:
tokenizer_v2 = SimpleTokenizerV2(word_to_index, index_to_word)
encoded_v2 = tokenizer_v2.encode("This word does not exist in the vocabulary")
print(encoded_v2)
decoded_v2 = tokenizer_v2.decode(encoded_v2)
print(decoded_v2)

[117, 1202, 1218, 776, 1218, 623, 1069, 1218]
This word <|unk|> not <|unk|> in the <|unk|>


##  Byte Pair Encoding

What is the BPE algorithm?

In [319]:
tokenizer = tiktoken.get_encoding("gpt2")
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)
strings = tokenizer.decode(integers)
print(strings)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]
Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


Second, the BPE tokenizer encodes and decodes unknown words, such as someunknownPlace, correctly. The BPE tokenizer can handle any unknown word. How does it achieve this without using <|unk|> tokens?
The algorithm underlying BPE breaks down words that aren’t in its predefined vocabulary into smaller subword units or even individual characters, enabling it to handle out-of-vocabulary words. 

## Data sampling

In [320]:
bpe_tokenizer = tiktoken.get_encoding("gpt2")
encoded_text = bpe_tokenizer.encode(raw_text)
print(f"Encoded text length: {len(encoded_text):,} tokens")

Encoded text length: 5,437 tokens


In [321]:
encoded_sample = encoded_text[50:]
context_size = 4         #1
for i in range(1, context_size+1):
    context = encoded_sample[:i]
    desired = encoded_sample[i]
    print(context, "---->", desired)

[257] ----> 7026
[257, 7026] ----> 15632
[257, 7026, 15632] ----> 438
[257, 7026, 15632, 438] ----> 2016


In [322]:
for i in range(1, context_size+1):
    context = encoded_sample[:i]
    desired = encoded_sample[i]
    print(bpe_tokenizer.decode(context), "---->", bpe_tokenizer.decode([desired]))

 a ---->  cheap
 a cheap ---->  genius
 a cheap genius ----> --
 a cheap genius-- ----> though


In [323]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

<span style="color: red">Redo the above logic in tensorflow</span>

In [324]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")                         #1
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)   #2
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,     #3
        num_workers=num_workers     #4
    )

    return dataloader

In [325]:
dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  464,  4643, 11600,   628]]), tensor([[ 4643, 11600,   628,   198]])]


In [326]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 4643, 11600,   628,   198]]), tensor([[11600,   628,   198,   197]])]


In [327]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=4, stride=4,
    shuffle=False
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[  464,  4643, 11600,   628],
        [  198,   197,   197,   197],
        [  197,   197,  7407,   342],
        [  854, 41328,   628,   628],
        [  198,   198,  1129,  2919],
        [  628,   628,   198,   198],
        [ 3109,  9213,   422, 11145],
        [  271,  1668,   319,  2805]])

Targets:
 tensor([[ 4643, 11600,   628,   198],
        [  197,   197,   197,   197],
        [  197,  7407,   342,   854],
        [41328,   628,   628,   198],
        [  198,  1129,  2919,   628],
        [  628,   198,   198,  3109],
        [ 9213,   422, 11145,   271],
        [ 1668,   319,  2805,  2534]])


## Creating token embeddings

  A model can't work with raw text like "cat". It needs numbers. But a  simple integer ID (e.g., cat = 4821) doesn't encode any meaning — the model has no way to know "cat" and "kitten" are related.
  What embeddings do

  Each token ID gets mapped to a vector of floats (e.g., 256 or 768 dimensions):

  "cat"    → [0.2, -0.5, 0.8, 0.1, ...]   # 768 numbers
  "kitten" → [0.19, -0.48, 0.79, 0.12, ...] # similar vector
  "car"    → [0.6,  0.3, -0.2, 0.9, ...]   # very different vector      

  Similar tokens end up with similar vectors — this is what makes them useful.

In [329]:
input_ids = torch.tensor([2, 3, 5, 1])
vocab_size = 6
output_dim = 3

In [330]:
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [ ]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)
